<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/v3/alignment-training/sft-smollm/sft-smollm.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("sft-smollm", ...)`


# Apply SFT on SmolLM

**Difficulty**: 🔴 Hard

---

### Problem Statement

A pretrained **base** model is a document completer. Ask it a question and it continues the document, usually with more questions, and it never stops. **Supervised fine-tuning (SFT)** turns it into an assistant that answers in a chat format and ends its turn.

Mechanically, SFT is next-token cross-entropy. Interviewers still ask about it, because almost every bug is in the **data path**, not the model:

- **Loss on the prompt.** Only the response should be trained, so the prompt positions get label `-100`.
- **Off-by-one.** The logits at position `t` predict the token at `t + 1`.
- **Padding.** Pads must be excluded from the loss, and real tokens must never be mistaken for pads.
- **End of turn.** If the end-of-turn token isn't in the labels, the model never learns to stop.
- **Normalization.** Averaging per-micro-batch means under gradient accumulation weights tokens unequally. Hugging Face fixed exactly this bug in `transformers` in late 2024.

You'll fine-tune **`HuggingFaceTB/SmolLM-135M`**, the base model whose architecture you built in the *SmolLM from Scratch* problem, on a tiny instruction dataset. Then you'll show that it follows the chat format on questions it never saw.

---

### Requirements

1. **`build_sft_example`**: prompt ids + response ids + end-of-turn id → `input_ids` and prompt-masked `labels`.
2. **`sft_collate`**: right-pad a list of examples into `input_ids`, `attention_mask` and `labels`.
3. **`sft_loss`**: the shifted, masked, token-level cross-entropy, with an optional global token count for gradient accumulation.
4. **`sft_train_step`**: one optimizer step over several micro-batches, normalized correctly.
5. **Fine-tune SmolLM-135M**: write `encode_chat_example`, train, and evaluate on held-out prompts.

---

### Constraints

- ✅ `labels` are aligned with `input_ids` (not pre-shifted), the Hugging Face convention. The shift happens inside the loss.
- ✅ Use `IGNORE_INDEX = -100` for positions that must not be trained.
- ✅ The loss is the mean over **all response tokens in the optimizer step**, however they're split into micro-batches.
- ❌ No `trl`, `SFTTrainer`, `Trainer` or HF data collators. Don't pass `labels=` to the HF model either, since it would compute the loss for you.

**Runtime:** about a minute on a laptop CPU after a ~270 MB model download; faster on a GPU or Apple MPS.

---

<details>
  <summary>💡 Hint</summary>

  - Build `labels` as a copy of `input_ids` and overwrite the prompt part with `-100`.
  - Padding `labels` with `-100` *in the collator* is what keeps a pad id that equals a real token id from erasing real labels.
  - `F.cross_entropy(..., ignore_index=-100, reduction="sum")` divided by the number of non-ignored targets is a token mean. With gradient accumulation, divide by the count over **all** micro-batches instead.

</details>

---

In [ ]:
import copy
import random
from types import SimpleNamespace

import torch
import torch.nn as nn
import torch.nn.functional as F

IGNORE_INDEX = -100   # F.cross_entropy's default ignore_index; Hugging Face uses the same value

torch.manual_seed(0)
random.seed(0)

## Part 1: Build one SFT example

An SFT example is the chat-formatted **prompt**, then the **response**, then the **end-of-turn** token. `labels` are a copy of `input_ids` with every prompt position set to `-100`:

```
input_ids : <|im_start|> user \n What is ... <|im_end|> \n <|im_start|> assistant \n | The capital ... . | <|im_end|>
labels    :     -100     -100 ...                                          ...  -100 | The capital ... . | <|im_end|>
```

Labels stay **aligned** with `input_ids`, and the shift happens in the loss. If `max_len` is given, truncate both from the right.

Accept Python lists or 1-D tensors, and return 1-D `torch.long` tensors.

In [ ]:
def build_sft_example(prompt_ids, response_ids, eos_id: int, max_len: int = None) -> dict:
    """
    Args:
        prompt_ids: token ids of the chat-formatted prompt (list or 1-D tensor).
        response_ids: token ids of the assistant's response (list or 1-D tensor).
        eos_id: end-of-turn token id, appended after the response and trained on.
        max_len: optional; truncate input_ids and labels to this many tokens.

    Returns:
        {"input_ids": LongTensor (L,), "labels": LongTensor (L,)} with prompt labels = IGNORE_INDEX.
    """
    # TODO: input_ids = prompt + response + [eos]
    # TODO: labels    = [IGNORE_INDEX] * len(prompt) + response + [eos]
    # TODO: truncate both to max_len if given
    # TODO: return both as torch.long tensors
    ...

In [ ]:
# === Part 1 tests ===
ex = build_sft_example([5, 6, 7], [8, 9], eos_id=2)
assert ex["input_ids"].tolist() == [5, 6, 7, 8, 9, 2], f"input_ids wrong: {ex['input_ids'].tolist()}"
assert ex["labels"].tolist() == [-100, -100, -100, 8, 9, 2], f"labels wrong: {ex['labels'].tolist()}"
assert ex["input_ids"].dtype == torch.long and ex["labels"].dtype == torch.long, "use torch.long"

cut = build_sft_example([5, 6, 7], [8, 9], eos_id=2, max_len=4)
assert cut["input_ids"].tolist() == [5, 6, 7, 8] and cut["labels"].tolist() == [-100, -100, -100, 8], \
    "truncate both tensors from the right"

from_tensors = build_sft_example(torch.tensor([5, 6, 7]), torch.tensor([8, 9]), eos_id=2)
assert from_tensors["labels"].tolist() == ex["labels"].tolist(), "tensor inputs must work too"
print("Part 1 passed ✓")

## Part 2: Collate a batch (right padding)

Examples have different lengths. Right-pad them to the longest one in the batch:

- `input_ids`: pad with `pad_id`
- `attention_mask`: 1 for real tokens, 0 for pads
- `labels`: pad with `IGNORE_INDEX`, **not** `pad_id`

The last point matters when `pad_id` is also a real token id. SmolLM's tokenizer has no pad token, so a common choice is to reuse `<|endoftext|>`. Many tokenizers reuse EOS. Because labels come from the examples and only the padding is `-100`, a genuine EOS in the labels survives.

In [ ]:
def sft_collate(examples: list, pad_id: int) -> dict:
    """
    Args:
        examples: list of {"input_ids": (L_i,), "labels": (L_i,)} from build_sft_example.
        pad_id: token id used to pad input_ids.

    Returns:
        {"input_ids": (B, L), "attention_mask": (B, L), "labels": (B, L)}, all torch.long,
        with L = max L_i and padding on the right.
    """
    # TODO: L = longest example
    # TODO: input_ids filled with pad_id, labels filled with IGNORE_INDEX, attention_mask with 0
    # TODO: copy each example into the left part of its row and set its mask to 1
    ...

In [ ]:
# === Part 2 tests ===
EOS = PAD = 2   # deliberately the same id
exs = [build_sft_example([5, 6], [7], EOS),            # length 4
       build_sft_example([5, 6, 7], [8, 9, 10], EOS),  # length 7
       build_sft_example([5], [], EOS)]                # length 2
batch = sft_collate(exs, pad_id=PAD)
assert batch["input_ids"].shape == (3, 7), f"expected (3, 7), got {tuple(batch['input_ids'].shape)}"
assert batch["attention_mask"].sum(1).tolist() == [4, 7, 2], "attention_mask must count the real tokens"
assert batch["input_ids"][0].tolist() == [5, 6, 7, 2, 2, 2, 2], "right-pad input_ids with pad_id"
assert batch["labels"][0].tolist() == [-100, -100, 7, 2, -100, -100, -100], \
    "pad labels with -100, and keep the real end-of-turn label even though it equals pad_id"
assert batch["labels"][2].tolist()[:2] == [-100, 2], "an empty response still trains the end-of-turn token"
assert all(v.dtype == torch.long for v in batch.values()), "all tensors should be torch.long"
print("Part 2 passed ✓")

## Part 3: The SFT loss

The logits at position `t` predict token `t + 1`, so drop the last logit and the first label:

```
logits[:, :-1]  vs  labels[:, 1:]
```

Cross-entropy over the non-ignored targets gives the **token mean**: every trained token counts equally, whichever sequence it's in.

**Gradient accumulation.** Suppose one optimizer step is split into micro-batches. If each one returns *its own* mean and you average those means, a micro-batch with 10 response tokens weighs as much as one with 1,000. Instead, compute the number of trained tokens across the **whole step** (`num_items_in_batch`, Hugging Face's name for it) and return `sum_of_token_losses / num_items_in_batch`. The micro-batch losses then **add up** to the full-batch token mean.

Also: upcast the logits to float32 before the softmax, because models often run in bf16. Return `0` (not `nan`) if nothing is trained.

In [ ]:
def sft_loss(logits: torch.Tensor, labels: torch.Tensor, num_items_in_batch: int = None) -> torch.Tensor:
    """
    Shifted next-token cross-entropy over non-ignored labels.

    Args:
        logits: (B, T, V) model outputs.
        labels: (B, T), aligned with the inputs; IGNORE_INDEX where nothing is trained.
        num_items_in_batch: if given, divide the summed loss by this (the trained-token count of the
            whole optimizer step) instead of by this batch's own count.

    Returns:
        Scalar loss.
    """
    # TODO: shift: logits[:, :-1] predict labels[:, 1:]
    # TODO: summed cross-entropy over all positions, ignoring IGNORE_INDEX (upcast logits to float32)
    # TODO: divide by num_items_in_batch, or by this batch's number of trained targets (at least 1)
    ...

In [ ]:
# === Part 3 tests ===
torch.manual_seed(0)
B, T, V = 3, 6, 13
logits = torch.randn(B, T, V)
labels = torch.randint(0, V, (B, T))
labels[0, :2] = IGNORE_INDEX           # prompt
labels[1, :4] = IGNORE_INDEX           # longer prompt
labels[2, 3:] = IGNORE_INDEX           # padding

# Loop oracle: position t predicts labels[t + 1]
total, count = 0.0, 0
for b in range(B):
    for t in range(T - 1):
        y = labels[b, t + 1].item()
        if y != IGNORE_INDEX:
            total += -F.log_softmax(logits[b, t], -1)[y]
            count += 1
oracle = total / count

got = sft_loss(logits, labels)
assert got.ndim == 0, "the loss must be a scalar"
assert torch.allclose(got, oracle, atol=1e-5), f"expected {oracle:.5f}, got {got:.5f}; check the shift"

# Logits whose next label is ignored (and the very last logit) must not matter
noisy = logits.clone()
unused = torch.zeros(B, T, dtype=torch.bool)
unused[:, :-1] = labels[:, 1:] == IGNORE_INDEX
unused[:, -1] = True
noisy[unused] = 100 * torch.randn(int(unused.sum()), V)
assert torch.allclose(sft_loss(noisy, labels), got, atol=1e-5), "ignored positions leaked into the loss"

# Gradient accumulation: micro-batch losses with a global count must add up to the full loss
n_total = int((labels[:, 1:] != IGNORE_INDEX).sum())
# (rows 0-1 hold 6 trained tokens, row 2 holds 2, so the two micro-batches are unequal)
accumulated = sft_loss(logits[:2], labels[:2], n_total) + sft_loss(logits[2:], labels[2:], n_total)
assert torch.allclose(accumulated, got, atol=1e-5), "with num_items_in_batch, micro-batch losses must sum to the full loss"
naive = (sft_loss(logits[:2], labels[:2]) + sft_loss(logits[2:], labels[2:])) / 2
print(f"full-batch token mean = {got:.4f}   accumulated (correct) = {accumulated:.4f}   mean of micro-batch means = {naive:.4f}")

all_ignored = torch.full((B, T), IGNORE_INDEX)
assert torch.isfinite(sft_loss(logits, all_ignored)), "a batch with nothing to train must not give nan"
print("Part 3 passed ✓")

## Part 4: One optimizer step with gradient accumulation

`micro_batches` is a list of collated batches that together form **one** optimizer step. Implement `sft_train_step`:

1. Zero the gradients.
2. Count the trained tokens across **all** micro-batches first. Remember the shift: it's the labels from position 1 onward that count.
3. For each micro-batch, run the forward pass, compute `sft_loss(..., num_items_in_batch=total)`, and call `backward()`. Gradients add up across calls.
4. Optionally clip the gradient norm, then `optimizer.step()`.
5. Return the step's loss (the sum of the micro-batch losses) as a Python float.

The model follows the Hugging Face contract: `model(input_ids=..., attention_mask=...)` returns an object with a `.logits` attribute. Move each batch to the model's device.

The test checks the property that matters: **splitting a batch into micro-batches must give exactly the same update as not splitting it.**

In [ ]:
def sft_train_step(model: nn.Module, micro_batches: list, optimizer: torch.optim.Optimizer,
                   max_grad_norm: float = None) -> float:
    """
    One optimizer step over several micro-batches.

    Args:
        model: returns an object with .logits of shape (B, T, V) from model(input_ids=..., attention_mask=...).
        micro_batches: list of dicts from sft_collate.
        optimizer: optimizer over the model's parameters.
        max_grad_norm: clip the total gradient norm to this, if given.

    Returns:
        The token-mean loss of the whole step, as a float.
    """
    # TODO: model.train(); find the model's device
    # TODO: zero the gradients
    # TODO: total trained tokens over ALL micro-batches (labels[:, 1:] != IGNORE_INDEX)
    # TODO: for each micro-batch: forward, sft_loss(..., num_items_in_batch=total), backward, accumulate the float loss
    # TODO: clip (if max_grad_norm), optimizer.step(), return the loss
    ...

In [ ]:
# === Part 4 tests ===
class TinyLM(nn.Module):
    """Embedding -> tanh -> Linear. Position-wise, so padding can't leak between tokens."""
    def __init__(self, vocab=20, dim=16):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim)
        self.head = nn.Linear(dim, vocab)

    def forward(self, input_ids, attention_mask=None):
        return SimpleNamespace(logits=self.head(torch.tanh(self.emb(input_ids))))


torch.manual_seed(0)
EOS, PAD = 19, 0
# Very different response lengths, so a wrong normalization shows up
exs = [build_sft_example([1, 2, 3], [4, 5, 6, 7, 8, 9, 10, 11], EOS),
       build_sft_example([1, 2], [12], EOS),
       build_sft_example([3, 4, 5, 6], [7, 8], EOS),
       build_sft_example([2], [13, 14, 15], EOS)]

model_split = TinyLM()
model_whole = copy.deepcopy(model_split)
reference = copy.deepcopy(model_split)
full = sft_collate(exs, PAD)
with torch.no_grad():
    expected_loss = sft_loss(reference(input_ids=full["input_ids"]).logits, full["labels"]).item()

loss_split = sft_train_step(model_split, [sft_collate(exs[:2], PAD), sft_collate(exs[2:], PAD)],
                            torch.optim.SGD(model_split.parameters(), lr=0.5))
loss_whole = sft_train_step(model_whole, [full], torch.optim.SGD(model_whole.parameters(), lr=0.5))

assert abs(loss_whole - expected_loss) < 1e-5, f"step loss {loss_whole:.5f} != full-batch loss {expected_loss:.5f}"
assert abs(loss_split - loss_whole) < 1e-5, \
    f"split into micro-batches the step loss is {loss_split:.5f}, unsplit {loss_whole:.5f}"
for (name, a), b in zip(model_split.named_parameters(), model_whole.parameters()):
    assert torch.allclose(a, b, atol=1e-6), \
        f"parameter '{name}' differs: micro-batching must give the same update as one big batch"

# And it trains
model = TinyLM()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
first = sft_train_step(model, [full], opt)
for _ in range(50):
    last = sft_train_step(model, [sft_collate(exs[:2], PAD), sft_collate(exs[2:], PAD)], opt, max_grad_norm=1.0)
assert last < 0.5 * first, f"loss should fall on a fixed batch ({first:.3f} -> {last:.3f})"
print(f"loss {first:.3f} -> {last:.3f}")
print("Part 4 passed ✓")

## Part 5: Fine-tune SmolLM-135M into a (tiny) assistant

**Chat format.** Use the ChatML format that SmolLM's instruct models use:

```
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
The capital of France is Paris.<|im_end|>
```

The base model's vocabulary already contains `<|im_start|>` and `<|im_end|>`, but the model was never trained to use them. Here `<|im_end|>` is the end-of-turn token, and `<|endoftext|>` is reused as the pad.

**Data.** 32 *"What is the capital of X?"* pairs for training, and 8 **held-out** countries for evaluation.

**Your task.** Write `encode_chat_example`: tokenize the formatted prompt and the response **separately** (so the label boundary is exact), with `add_special_tokens=False`, and pass them to `build_sft_example`. Then run the cells below.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "HuggingFaceTB/SmolLM-135M"     # the *base* model, not the instruct one
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(device)
IM_END = tokenizer.convert_tokens_to_ids("<|im_end|>")
PAD_ID = tokenizer.convert_tokens_to_ids("<|endoftext|>")
print(f"device={device}  params={sum(p.numel() for p in model.parameters()) / 1e6:.0f}M  "
      f"<|im_end|>={IM_END}  pad=<|endoftext|>={PAD_ID}")


def chat_prompt(question: str) -> str:
    return f"<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"


CAPITALS = [
    ("France", "Paris"), ("Germany", "Berlin"), ("Italy", "Rome"), ("Spain", "Madrid"),
    ("Japan", "Tokyo"), ("China", "Beijing"), ("India", "New Delhi"), ("Russia", "Moscow"),
    ("Canada", "Ottawa"), ("Australia", "Canberra"), ("Egypt", "Cairo"), ("Kenya", "Nairobi"),
    ("Brazil", "Brasilia"), ("Argentina", "Buenos Aires"), ("Mexico", "Mexico City"), ("Peru", "Lima"),
    ("Chile", "Santiago"), ("Greece", "Athens"), ("Portugal", "Lisbon"), ("Poland", "Warsaw"),
    ("Norway", "Oslo"), ("Sweden", "Stockholm"), ("Finland", "Helsinki"), ("Denmark", "Copenhagen"),
    ("Ireland", "Dublin"), ("Austria", "Vienna"), ("Hungary", "Budapest"), ("Turkey", "Ankara"),
    ("Iran", "Tehran"), ("Thailand", "Bangkok"), ("Vietnam", "Hanoi"), ("South Korea", "Seoul"),
    # held out: never trained on
    ("Netherlands", "Amsterdam"), ("Belgium", "Brussels"), ("Switzerland", "Bern"), ("Cuba", "Havana"),
    ("Nigeria", "Abuja"), ("Morocco", "Rabat"), ("Indonesia", "Jakarta"), ("Ukraine", "Kyiv"),
]
train_pairs, heldout_pairs = CAPITALS[:32], CAPITALS[32:]


def qa(country: str, capital: str) -> tuple:
    return f"What is the capital of {country}?", f"The capital of {country} is {capital}."

In [ ]:
def encode_chat_example(question: str, answer: str, max_len: int = 64) -> dict:
    """Chat-format one (question, answer) pair and build its SFT example."""
    # TODO: prompt_ids = tokenizer(chat_prompt(question), add_special_tokens=False).input_ids
    # TODO: response_ids = tokenizer(answer, add_special_tokens=False).input_ids
    # TODO: return build_sft_example(prompt_ids, response_ids, IM_END, max_len)
    ...

In [ ]:
# === Look at what is actually trained ===
ex = encode_chat_example(*qa("France", "Paris"))
trained = ex["labels"][ex["labels"] != IGNORE_INDEX]
print("full input :", repr(tokenizer.decode(ex["input_ids"])))
print("trained on :", repr(tokenizer.decode(trained)))
assert tokenizer.decode(trained) == "The capital of France is Paris.<|im_end|>", \
    "only the response and the end-of-turn token should be trained"


@torch.no_grad()
def greedy_answer(model, question: str, max_new_tokens: int = 30) -> tuple:
    """Greedy decoding until <|im_end|>. Returns (text, stopped)."""
    model.eval()
    ids = torch.tensor([tokenizer(chat_prompt(question), add_special_tokens=False).input_ids], device=device)
    out = []
    for _ in range(max_new_tokens):
        next_id = model(input_ids=ids).logits[0, -1].argmax().item()
        if next_id == IM_END:
            return tokenizer.decode(out), True
        out.append(next_id)
        ids = torch.cat([ids, torch.tensor([[next_id]], device=device)], dim=1)
    return tokenizer.decode(out), False


def evaluate(model, pairs) -> list:
    return [(country, capital, *greedy_answer(model, qa(country, capital)[0])) for country, capital in pairs]


before = evaluate(model, heldout_pairs)
print("\n=== Base model, before SFT ===")
for country, _, text, stopped in before[:3]:
    print(f"{country:>12}: {text[:70]!r}  stopped={stopped}")

In [ ]:
# === Fine-tune ===
torch.manual_seed(0)
random.seed(0)
train_examples = [encode_chat_example(*qa(c, k)) for c, k in train_pairs]

EPOCHS, BATCH, MICRO = 3, 8, 4        # 2 micro-batches of 4 per optimizer step
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.0)
losses = []
for epoch in range(EPOCHS):
    random.shuffle(train_examples)
    for i in range(0, len(train_examples), BATCH):
        batch = train_examples[i:i + BATCH]
        micro_batches = [sft_collate(batch[j:j + MICRO], PAD_ID) for j in range(0, len(batch), MICRO)]
        losses.append(sft_train_step(model, micro_batches, optimizer, max_grad_norm=1.0))
    print(f"epoch {epoch}: step losses {[round(l, 3) for l in losses[-4:]]}")

In [ ]:
# === Validation ===
after = evaluate(model, heldout_pairs)
print("=== After SFT, on held-out countries ===")
for country, capital, text, stopped in after:
    print(f"{country:>12}: {text!r}  stopped={stopped}  {'✓' if capital in text else '✗ ' + capital}")

n = len(heldout_pairs)
stop_before = sum(s for *_, s in before) / n
stop_after = sum(s for *_, s in after) / n
formatted = sum(text.startswith("The capital of ") and text.endswith(".") for *_, text, _ in after) / n
correct = sum(k in text for _, k, text, _ in after) / n
print(f"\nends its turn : {stop_before:.0%} -> {stop_after:.0%}")
print(f"follows format: {formatted:.0%}")
print(f"right capital : {correct:.0%}  (knowledge comes from pretraining, not from SFT)")

# Test 1: training loss should drop substantially
print("\n=== Loss Test ===")
print(f"first step {losses[0]:.3f} -> last step {losses[-1]:.3f}")
assert losses[-1] < 0.5 * losses[0], "SFT loss should at least halve"
print("PASSED")

# Test 2: the model learned to end its turn
print("=== End-of-Turn Test ===")
assert stop_after >= 0.75 and stop_after > stop_before, "after SFT the model should emit <|im_end|> on held-out prompts"
print("PASSED")

# Test 3: the format generalizes to unseen prompts
print("=== Format Test ===")
assert formatted >= 0.75, "held-out answers should follow the trained format"
print("PASSED")

print("\nAll tests passed!")

## Interview follow-ups

1. What happens if you *don't* mask the prompt? Is that ever a good idea? How would you mask a multi-turn conversation?
2. You trained without the end-of-turn token in the labels. What does the model do at inference time?
3. Your tokenizer has no pad token, so you set `pad_token = eos_token` and build labels as `input_ids` with pad positions set to `-100`. What silently breaks?
4. Why right-pad for training but left-pad for batched generation?
5. **Packing:** you concatenate many short examples into one 4k-token row to avoid padding. What must you change in attention and position ids, and why?
6. Gradient accumulation with 4 micro-batches on each of 8 GPUs. How do you get the token-level normalization exactly right across devices?
7. On the held-out set the model may answer "The capital of Switzerland is Geneva." What did SFT teach, and what didn't it teach? What happens if you fine-tune on facts the base model doesn't know?
8. Which KL divergence does SFT minimize? (See the KL divergence problem.)
9. Full fine-tuning vs LoRA for this: memory, quality, and when you'd pick each.